# Self-RAG — Simplest LangChain + ChromaDB Implementation

This notebook keeps the Self-RAG logic from `self_rag_step7.ipynb`, but removes LangGraph and uses simple LangChain chains plus normal Python `if` / `while` control flow.

### Replacements
- OpenAI LLM → Groq `ChatGroq`
- OpenAI embeddings → `HuggingFaceEndpointEmbeddings`
- FAISS → ChromaDB
- LangGraph → normal Python orchestration

### Self-RAG flow preserved
`Decide Retrieval → Direct Answer OR Retrieve → Relevance → Generate → IsSUP → Revise Loop → IsUSE → Useful END / Rewrite Query → Retrieve Again / No Answer Found`

In [1]:
from typing import List, Literal
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEndpointEmbeddings
from langchain_groq import ChatGroq
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()

HF_TOKEN = os.getenv("HF_TOKEN")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN is missing")

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY is missing")

C:\Users\Uttam\AppData\Local\Temp\ipykernel_2260\300839020.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


## 1. Load, split and store documents

In [2]:
docs = (
    PyPDFLoader("./documents/Company_Policies.pdf").load()
    + PyPDFLoader("./documents/Company_Profile.pdf").load()
    + PyPDFLoader("./documents/Product_and_Pricing.pdf").load()
)

chunks = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=150,
).split_documents(docs)

embeddings = HuggingFaceEndpointEmbeddings(
    model="sentence-transformers/all-MiniLM-L6-v2",
    huggingfacehub_api_token=HF_TOKEN,
)

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="self_rag_demo",
)

retriever = vector_store.as_retriever(
    search_kwargs={"k": 4}
)

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    api_key=GROQ_API_KEY,
)

print(f"Pages loaded: {len(docs)}")
print(f"Chunks created: {len(chunks)}")
print("Vector store: ChromaDB")

Pages loaded: 8
Chunks created: 16
Vector store: ChromaDB


## 2. Decide whether retrieval is needed

In [3]:
class RetrieveDecision(BaseModel):
    should_retrieve: bool = Field(
        ...,
        description="True if external documents are needed to answer reliably, else False."
    )

decide_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You decide whether retrieval is needed.\n"
        "Return JSON with key: should_retrieve (boolean).\n\n"
        "Guidelines:\n"
        "- should_retrieve=True if answering requires specific facts from company documents.\n"
        "- should_retrieve=False for general explanations/definitions.\n"
        "- If unsure, choose True."
    ),
    ("human", "Question: {question}"),
])

decide_chain = decide_prompt | llm.with_structured_output(
    RetrieveDecision,
    method="json_schema",
    strict=True
)

## 3. Direct generation (when retrieval is not needed)

In [4]:
direct_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Answer using only your general knowledge.\n"
        "If it requires specific company info, say:\n"
        "'I don't know based on my general knowledge.'"
    ),
    ("human", "{question}"),
])

direct_chain = direct_prompt | llm

## 4. Relevance check for retrieved documents

In [5]:
class RelevanceDecision(BaseModel):
    is_relevant: bool = Field(
        ...,
        description="True only when the document is relevant to the question; for entity-specific questions, the document must concern the same entity."
    )

relevance_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are judging whether the retrieved document is relevant to the user's question.\n"
        "Return JSON matching the schema.\n\n"

        "Rules:\n"
        "1. A document is relevant only if it contains information that can help answer the question.\n"
        "2. If the question asks about a specific named entity such as a company, person, product, or organization, "
        "the document must refer to that same entity.\n"
        "3. Matching only the general topic is NOT enough for entity-specific questions.\n"
        "   Example: a NexaAI pricing document is NOT relevant to a question about Microsoft's pricing.\n"
        "4. For general non-entity questions, topic-level relevance is acceptable.\n"
        "5. Do NOT decide whether the document fully answers the question. "
        "Only decide whether it is relevant enough to use.\n"
        "6. When there is a clear entity mismatch, return is_relevant=false.\n"
    ),
    ("human", "Question:\n{question}\n\nDocument:\n{document}"),
])

relevance_chain = relevance_prompt | llm.with_structured_output(
    RelevanceDecision,
    method="json_schema",
    strict=True
)

## 5. Generate from relevant context

In [6]:
rag_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a business rag chatbot.\n\n"
        "You will receive a CONTEXT block from internal company documents.\n"
        "Task:\n"
        "Answer the question based on the context"
        "Dont mention that you are getting a context in your answer"
    ),
    ("human", "Question:\n{question}\n\nContext:\n{context}"),
])

rag_chain = rag_prompt | llm

## 6. IsSUP — verify grounding/support

The answer can be `fully_supported`, `partially_supported`, or `no_support`. Up to 3 short evidence quotes are returned.

In [7]:
class IsSUPDecision(BaseModel):
    issup: Literal[
        "fully_supported",
        "partially_supported",
        "no_support",
    ]
    evidence: List[str] = Field(
    ...,
    description="Short evidence excerpts supporting the answer. Return an empty list when no evidence is available."
)

issup_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are verifying whether the ANSWER is supported by the CONTEXT.\n"
        "Return JSON with keys: issup, evidence.\n"
        "issup must be one of: fully_supported, partially_supported, no_support.\n\n"
        "How to decide issup:\n"
        "- fully_supported:\n"
        "  Every meaningful claim is explicitly supported by CONTEXT, and the ANSWER does NOT introduce\n"
        "  any qualitative/interpretive words that are not present in CONTEXT.\n"
        "  (Examples of disallowed words unless present in CONTEXT: culture, generous, robust, designed to,\n"
        "  supports professional development, best-in-class, employee-first, etc.)\n\n"
        "- partially_supported:\n"
        "  The core facts are supported, BUT the ANSWER includes ANY abstraction, interpretation, or qualitative\n"
        "  phrasing not explicitly stated in CONTEXT (e.g., calling policies 'culture', saying leave is 'generous',\n"
        "  or inferring outcomes like 'supports professional development').\n\n"
        "- no_support:\n"
        "  The key claims are not supported by CONTEXT.\n\n"
        "Rules:\n"
        "- Be strict: if you see ANY unsupported qualitative/interpretive phrasing, choose partially_supported.\n"
        "- If the answer is mostly unrelated to the question or unsupported, choose no_support.\n"
        "- Evidence: include up to 3 short excerpts from CONTEXT that support the supported parts.Do NOT put quotation marks around the evidence.\n"
        "- Do not use outside knowledge."
    ),
    (
        "human",
        "Question:\n{question}\n\n"
        "Answer:\n{answer}\n\n"
        "Context:\n{context}\n"
    ),
])

issup_chain = issup_prompt | llm.with_structured_output(
    IsSUPDecision,
    method="json_schema",
    strict=True
)

MAX_RETRIES = 10

## 7. Revise unsupported answer

In [8]:
revise_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a STRICT reviser.\n\n"
        "You must output based on the following format:\n\n"
        "FORMAT (quote-only answer):\n"
        "- <direct quote from the CONTEXT>\n"
        "- <direct quote from the CONTEXT>\n\n"
        "Rules:\n"
        "- Use ONLY the CONTEXT.\n"
        "- Do NOT add any new words besides bullet dashes and the quotes themselves.\n"
        "- Do NOT explain anything.\n"
        "- Do NOT say 'context', 'not mentioned', 'does not mention', 'not provided', etc.\n"
    ),
    (
        "human",
        "Question:\n{question}\n\n"
        "Current Answer:\n{answer}\n\n"
        "CONTEXT:\n{context}"
    ),
])

revise_chain = revise_prompt | llm

## 8. IsUSE — check usefulness

In [9]:
class IsUSEDecision(BaseModel):
    isuse: Literal["useful", "not_useful"]
    reason: str = Field(..., description="Short reason in 1 line.")

isuse_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are judging whether the ANSWER completely satisfies the QUESTION.\n\n"

        "Goal:\n"
        "Decide whether the answer addresses ALL information explicitly requested by the user.\n\n"

        "Return JSON with keys: isuse, reason.\n"
        "isuse must be one of: useful, not_useful.\n\n"

        "Rules:\n"
        "- useful: The answer fully addresses all parts of the question.\n"
        "- not_useful: Any requested part of the question is missing, unanswered, or explicitly unavailable.\n"
        "- If the question has multiple parts, ALL parts must be answered for useful=true.\n"
        "- An answer that says a requested detail is unavailable is NOT useful when that detail was explicitly requested.\n"
        "- Do NOT use outside knowledge.\n"
        "- Do NOT re-check grounding. IsSUP already handled grounding.\n"
        "- Only judge whether the user's question was fully answered.\n"
        "- Keep reason to 1 short line."
    ),
    ("human", "Question:\n{question}\n\nAnswer:\n{answer}"),
])

isuse_chain = isuse_prompt | llm.with_structured_output(
    IsUSEDecision,
    method="json_schema",
    strict=True
)

MAX_REWRITE_TRIES = 3

## 9. Rewrite the retrieval query

In [10]:
class RewriteDecision(BaseModel):
    retrieval_query: str = Field(
        ...,
        description="Rewritten query optimized for vector retrieval against internal company PDFs."
    )

rewrite_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Rewrite the user's QUESTION into a query optimized for vector retrieval over INTERNAL company PDFs.\n\n"
        "Rules:\n"
        "- Keep it short (6–16 words).\n"
        "- Preserve key entities (e.g., NexaAI, plan names).\n"
        "- Add 2–5 high-signal keywords that likely appear in policy/pricing docs.\n"
        "- Remove filler words.\n"
        "- Do NOT answer the question.\n"
        "- Output JSON with key: retrieval_query\n\n"
        "Examples:\n"
        "Q: 'Do NexaAI plans include a free trial?'\n"
        "-> {{'retrieval_query': 'NexaAI free trial duration trial period plans'}}\n\n"
        "Q: 'What is NexaAI refund policy?'\n"
        "-> {{'retrieval_query': 'NexaAI refund policy cancellation refund timeline charges'}}"
    ),
    (
        "human",
        "QUESTION:\n{question}\n\n"
        "Previous retrieval query:\n{retrieval_query}\n\n"
        "Answer (if any):\n{answer}"
    ),
])

rewrite_chain = rewrite_prompt | llm.with_structured_output(
    RewriteDecision,
    method="json_schema",
    strict=True
)

## 10. Complete Self-RAG pipeline

Only one user input is required: `question`. `retrieval_query`, retry counters, documents, context, and verification results are internal.

In [11]:
def self_rag(question: str) -> dict:
    # Internal state
    retrieval_query = question
    rewrite_tries = 0
    rewrite_history = []
    retries = 0
    docs = []
    relevant_docs = []
    context = ""
    answer = ""
    issup = "no_support"
    evidence = []
    isuse = "not_useful"
    use_reason = ""

    # 1) Decide retrieval
    decision = decide_chain.invoke({"question": question})
    need_retrieval = decision.should_retrieve

    # 2) No retrieval -> direct answer -> END
    if not need_retrieval:
        answer = direct_chain.invoke({"question": question}).content
        return {
            "question": question,
            "need_retrieval": False,
            "retrieval_query": retrieval_query,
            "answer": answer,
        }

    # 3) Retrieval loop
    while True:
        docs = retriever.invoke(retrieval_query)

        # 4) Relevance filtering
        relevant_docs = []
        for doc in docs:
            result = relevance_chain.invoke({
                "question": question,
                "document": doc.page_content,
            })

            if result.is_relevant:
                relevant_docs.append(doc)

        # No relevant documents -> END
        if not relevant_docs:
            return {
                "question": question,
                "need_retrieval": True,
                "retrieval_query": retrieval_query,
                "rewrite_tries": rewrite_tries,
                "docs": docs,
                "relevant_docs": [],
                "answer": "No answer found.",
            }

        # 5) Generate answer from context
        context = "\n\n---\n\n".join(
            doc.page_content for doc in relevant_docs
        ).strip()

        answer = rag_chain.invoke({
            "question": question,
            "context": context,
        }).content

        # 6) IsSUP -> revise -> IsSUP
        while True:
            support = issup_chain.invoke({
                "question": question,
                "answer": answer,
                "context": context,
            })

            issup = support.issup
            evidence = support.evidence

            if issup == "fully_supported" or retries >= MAX_RETRIES:
                break

            answer = revise_chain.invoke({
                "question": question,
                "answer": answer,
                "context": context,
            }).content

            retries += 1

        # 7) IsUSE
        usefulness = isuse_chain.invoke({
            "question": question,
            "answer": answer,
        })

        isuse = usefulness.isuse
        use_reason = usefulness.reason

        if isuse == "useful":
            return {
                "question": question,
                "need_retrieval": True,
                "retrieval_query": retrieval_query,
                "rewrite_tries": rewrite_tries,
                "rewrite_history": rewrite_history,
                "docs": docs,
                "relevant_docs": relevant_docs,
                "context": context,
                "answer": answer,
                "issup": issup,
                "evidence": evidence,
                "retries": retries,
                "isuse": isuse,
                "use_reason": use_reason,
            }

        # Not useful + rewrite limit reached -> END
        if rewrite_tries >= MAX_REWRITE_TRIES:
            return {
                "question": question,
                "need_retrieval": True,
                "retrieval_query": retrieval_query,
                "rewrite_tries": rewrite_tries,
                "rewrite_history": rewrite_history,
                "docs": docs,
                "relevant_docs": relevant_docs,
                "answer": "No answer found.",
                "issup": issup,
                "evidence": evidence,
                "retries": retries,
                "isuse": isuse,
                "use_reason": use_reason,
            }

        # 8) Rewrite retrieval query and start retrieval again
        rewritten = rewrite_chain.invoke({
            "question": question,
            "retrieval_query": retrieval_query,
            "answer": answer,
        })

        retrieval_query = rewritten.retrieval_query
        rewrite_tries += 1

        rewrite_history.append({
            "rewrite": rewrite_tries,
            "query": retrieval_query,
        })

        # Reset retrieval-side state before next attempt
        docs = []
        relevant_docs = []
        context = ""
        # Loop back to Retrieve

## 11. Run

Only the user's question is supplied.

In [12]:
# result = self_rag(
#     "What is NexaAI's annual leave policy, and what percentage refund does NexaAI provide when a customer cancels a plan?"
# )

# print("\n===== SELF-RAG RESULT =====\n")
# print("Question:", result.get("question"))
# print("Need Retrieval:", result.get("need_retrieval"))
# print("Retrieval Query:", result.get("retrieval_query"))
# print("Rewrite Tries:", result.get("rewrite_tries", 0))
# print("\nRewritten Queries:")

# for item in result.get("rewrite_history", []):
#     print(f"{item['rewrite']}. {item['query']}")
    
# print("Support Revision Tries:", result.get("retries", 0))
# print("IsSUP:", result.get("issup", "N/A"))
# print("Evidence:", result.get("evidence", []))
# print("IsUSE:", result.get("isuse", "N/A"))
# print("Usefulness Reason:", result.get("use_reason", ""))
# print("\nFinal Answer:\n")
# print(result.get("answer"))
# print("\n==========================\n")

## 12. Automated Self-RAG Testing

In [13]:
# ==========================================
# 11. Automated Self-RAG Testing
# ==========================================

queries = [
    # Query 1
    "What is machine learning?",

    # Query 2
    "What is NexaAI's annual leave policy?",

    # Query 3
    "What is Microsoft's Copilot pricing per user per month?",

    # Query 4
    "Where is NexaAI headquartered and when was it founded?",

    # Query 5
    "Describe NexaAI's company culture.",

    # Query 6
    "What is NexaAI's annual leave policy, and what percentage refund does NexaAI provide when a customer cancels a plan?",
]


def format_result(result):
    lines = []

    lines.append("===== SELF-RAG RESULT =====")
    lines.append("")
    lines.append(f"Question: {result.get('question')}")
    lines.append(f"Need Retrieval: {result.get('need_retrieval')}")
    lines.append(f"Retrieval Query: {result.get('retrieval_query')}")
    lines.append(f"Rewrite Tries: {result.get('rewrite_tries', 0)}")

    rewrite_history = result.get("rewrite_history", [])

    if rewrite_history:
        lines.append("")
        lines.append("Rewritten Queries:")

        for item in rewrite_history:
            lines.append(
                f"{item['rewrite']}. {item['query']}"
            )

    lines.append(
        f"Support Revision Tries: {result.get('retries', 0)}"
    )
    lines.append(
        f"IsSUP: {result.get('issup', 'N/A')}"
    )
    lines.append(
        f"Evidence: {result.get('evidence', [])}"
    )
    lines.append(
        f"IsUSE: {result.get('isuse', 'N/A')}"
    )
    lines.append(
        f"Usefulness Reason: {result.get('use_reason', '')}"
    )

    lines.append("")
    lines.append("Final Answer:")
    lines.append("")
    lines.append(result.get("answer", ""))

    lines.append("")
    lines.append("==========================")

    return "\n".join(lines)


# Run all queries and write results to output.txt
with open("output.txt", "w", encoding="utf-8") as f:

    for query in queries:

        try:
            result = self_rag(query)

            output = format_result(result)

            # Display in notebook
            print(output)
            print("\n")

            # Save to file
            f.write(output)
            f.write("\n\n")

        except Exception as e:

            error_output = (
                "===== SELF-RAG ERROR =====\n\n"
                f"Question: {query}\n"
                f"Error: {type(e).__name__}: {e}\n"
                "=========================="
            )

            print(error_output)
            print("\n")

            f.write(error_output)
            f.write("\n\n")


print("All 6 queries completed.")
print("Results saved to: output.txt")

===== SELF-RAG RESULT =====

Question: What is machine learning?
Need Retrieval: False
Retrieval Query: What is machine learning?
Rewrite Tries: 0
Support Revision Tries: 0
IsSUP: N/A
Evidence: []
IsUSE: N/A
Usefulness Reason: 

Final Answer:

Machine learning is a subfield of artificial intelligence that focuses on developing algorithms and statistical models that enable computers to learn from data and improve their performance on a specific task without being explicitly programmed for every possible scenario. Instead of following fixed, hand‑crafted rules, a machine‑learning system identifies patterns, makes predictions, or makes decisions by analyzing examples (training data) and then generalizing that knowledge to new, unseen data. Common approaches include supervised learning (learning from labeled examples), unsupervised learning (discovering structure in unlabeled data), and reinforcement learning (learning through trial‑and‑error interactions with an environment).



===== SEL

## 12. Simplified Self-RAG flow

```text
Question
   ↓
Decide Retrieval
   ├── No → Direct Answer → END
   │
   └── Yes
        ↓
     Chroma Retrieve
        ↓
     Relevance Check
        ├── No relevant docs → No Answer Found → END
        │
        └── Relevant docs
             ↓
         Generate Answer
             ↓
           IsSUP
            ├── Fully Supported → IsUSE
            └── Partial / No Support → Revise → IsSUP
                                        (max 10 revisions)
                                             ↓
                                           IsUSE
                                            ├── Useful → END
                                            └── Not Useful
                                                 ├── max 3 rewrites → No Answer Found
                                                 └── Rewrite Retrieval Query
                                                        ↓
                                                     Retrieve Again ↺
```

**Key point:** `retrieval_query` is generated internally only when Self-RAG decides the previous answer was not useful. The user supplies only the question.